# Lección 2 · ¿Cuánto vale una hora más?

**Qué vas a aprender**
- **Precio sombra**: cuánto sube la ganancia con una unidad más de un recurso.
- Por qué un recurso que sobra vale 0.
- Hasta dónde vale el precio sombra: su **rango**.
- **Costo reducido**: cuánto le falta a un producto para entrar al plan.
- El **problema dual**: el mismo modelo visto desde el que compra tus horas.
- Pasar de 2 productos a 30 × 8 con los datos en tablas (pandas), y leer el top 5 de cada pieza.

**Cómo usar este notebook**
- Corre las celdas en orden con `Shift + Enter`.
- Antes de cada celda, adivina qué va a imprimir.

## 0. Preparar el entorno
Igual que en la lección 1.

In [ ]:
%pip install -q pyomo highspy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyomo.environ as pyo


def eur(x):
    return f"{x:,.0f}".replace(",", ".") + " €"

## 1. Dónde quedamos
En la lección 1 subiste el mecanizado de 100 h a 101 h y la ganancia pasó de 18.000 € a 18.100 €.
Una hora más vale **100 €**.

Para saberlo resolviste el modelo dos veces. Con 8 máquinas y 30 productos serían decenas de corridas.
El solver ya calcula ese número al resolver. Solo hay que pedírselo.

**Analogía.** Tu hijo duerme la siesta y tú estudias polaco. Una hora más de niñera vale mucho:
la usas toda para estudiar. Pero si ya tienes horas libres que no usas, una hora más de niñera vale 0.
Eso es el precio sombra: lo que vale **una unidad más** de un recurso.

## 2. El taller, ahora con precios sombra
Es el mismo modelo de la lección 1, dentro de una función. La única novedad son dos *suffix*:
- Un **suffix** es una etiqueta extra que el solver pega a cada restricción o variable.
- `m.dual` guarda el **precio sombra** de cada restricción (en inglés: *dual value* o *shadow price*).
- `m.rc` guarda el **costo reducido** de cada variable (*reduced cost*). Lo usamos en la sección 5.

In [ ]:
GANANCIA = {"valvula": 300, "conector": 200}
HORAS = {("valvula", "mecanizado"): 2, ("valvula", "prueba"): 1,
         ("conector", "mecanizado"): 1, ("conector", "prueba"): 1}
CAPACIDAD = {"mecanizado": 100, "prueba": 80}


def taller(ganancia=GANANCIA, horas=HORAS, capacidad=CAPACIDAD, max_valvulas=40):
    m = pyo.ConcreteModel()
    m.P = pyo.Set(initialize=list(ganancia))
    m.M = pyo.Set(initialize=list(capacidad))
    m.x = pyo.Var(m.P, within=pyo.NonNegativeReals)
    m.obj = pyo.Objective(expr=sum(ganancia[p] * m.x[p] for p in m.P), sense=pyo.maximize)
    m.cap = pyo.Constraint(m.M, rule=lambda m, k: sum(horas.get((p, k), 0) * m.x[p] for p in m.P) <= capacidad[k])
    m.mercado = pyo.Constraint(expr=m.x["valvula"] <= max_valvulas)
    m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)  # precios sombra
    m.rc = pyo.Suffix(direction=pyo.Suffix.IMPORT)    # costos reducidos
    pyo.SolverFactory("appsi_highs").solve(m)
    return m


m = taller()
print("Plan:", {p: pyo.value(m.x[p]) for p in m.P}, "→", eur(pyo.value(m.obj)))

Ahora la tabla de la lección 1 con una columna nueva: el precio sombra.

In [ ]:
def tabla_restricciones(m):
    filas = [(f"cap[{k}]", m.cap[k]) for k in m.M] + [("mercado", m.mercado)]
    datos = []
    for nombre, r in filas:
        usado, limite = pyo.value(r.body), pyo.value(r.upper)
        datos.append({"restriccion": nombre, "usado": usado, "limite": limite,
                      "holgura": limite - usado, "precio_sombra": m.dual[r]})
    return pd.DataFrame(datos)


tabla_restricciones(m)

**Cómo se lee:**
- Mecanizado: 100 €. Una hora más sube la ganancia 100 €. Es lo que mediste a mano en la lección 1.
- Prueba: 100 €. Lo viste en el ejercicio 2: 10 horas menos costaron 1.000 €.
- Mercado: 0 €. Sobran 20 válvulas de demanda; una más no cambia nada.

**Regla:** si una restricción tiene holgura, su precio sombra es 0. Si su precio sombra no es 0, está activa.
Se llama **holgura complementaria** (*complementary slackness*). Es la niñera que sobra.

**Nota de signos.** Aquí salen positivos. Con otros solvers o con restricciones `≥` pueden salir negativos.
Léelos siempre como: "¿cuánto cambia el objetivo si el lado derecho sube 1?".

### Comprobarlo
Un número que vas a llevarle al jefe se comprueba. Resolvemos con 1 hora más en cada máquina.

In [ ]:
base = pyo.value(taller().obj)
for k in CAPACIDAD:
    nueva = pyo.value(taller(capacidad={**CAPACIDAD, k: CAPACIDAD[k] + 1}).obj)
    print(f"{k:<11} +1 h → ganancia {eur(nueva)} (sube {eur(nueva - base)})")
nueva = pyo.value(taller(max_valvulas=41).obj)
print(f"mercado     +1   → ganancia {eur(nueva)} (sube {eur(nueva - base)})")

## 3. ¿Hasta dónde vale 100 € la hora?
El precio sombra vale **dentro de un rango**. Si sigues agregando horas de mecanizado,
llega un punto en que otra restricción manda y la hora extra ya no sirve.

**Analogía.** La primera hora de niñera la usas para polaco. La décima ya no: estás cansado
y no tienes nada más que hacer. El valor de la hora baja por escalones.

Vamos a resolver el modelo para cada capacidad de mecanizado de 40 h a 180 h y dibujar la ganancia.

In [ ]:
horas_mec = list(range(40, 181))
filas = []
for h in horas_mec:
    mh = taller(capacidad={"mecanizado": h, "prueba": 80})
    filas.append({"mecanizado_h": h, "ganancia": pyo.value(mh.obj), "precio_sombra": mh.dual[mh.cap["mecanizado"]]})
barrido = pd.DataFrame(filas)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(barrido["mecanizado_h"], barrido["ganancia"])
ax.axvline(100, color="gray", linestyle="--", label="hoy: 100 h")
ax.set(xlabel="Horas de mecanizado", ylabel="Ganancia (€)", title="Ganancia según horas de mecanizado")
ax.legend()
plt.show()

In [ ]:
# Cada tramo con el mismo precio sombra es un escalón
tramos = barrido.groupby("precio_sombra")["mecanizado_h"].agg(["min", "max"]).sort_values("min")
tramos

**Lectura:**
- De 40 a 80 h, cada hora vale 200 €: solo conviene hacer conectores (200 € por hora de mecanizado, contra 150 € de la válvula).
- De 80 a 120 h, cada hora vale 100 €. Hoy estamos aquí (100 h).
- Desde 120 h, cada hora vale 0 €: ya fabricas 40 válvulas, el máximo del mercado, y manda la prueba de presión.

La pendiente de la curva es el precio sombra. En los quiebres el óptimo salta a otra **esquina**,
como viste en la lección 1. Por eso el precio sombra solo vale dentro de su tramo.

**Para el jefe:** "Cada hora extra de mecanizado vale 100 €, hasta 20 horas extra. Más allá, no vale nada."

## 4. El problema dual: el que quiere comprarte las horas
Un taller vecino quiere **comprarte todas tus horas** de mecanizado y de prueba, y tu cupo de mercado de válvulas.
Te ofrece un precio por hora. ¿Cuál es el precio mínimo que aceptarías?

Aceptas solo si por cada producto te paga al menos lo que ganarías fabricándolo:
- Una válvula usa 2 h de mecanizado, 1 h de prueba y 1 cupo de mercado: 2·y_mec + y_pru + y_mer ≥ 300.
- Un conector usa 1 h de mecanizado y 1 h de prueba: y_mec + y_pru ≥ 200.

El vecino quiere pagar lo menos posible: minimizar 100·y_mec + 80·y_pru + 40·y_mer.

Eso es otro programa lineal, el **dual**. Resolvámoslo.

In [ ]:
d = pyo.ConcreteModel()
d.y = pyo.Var(["mecanizado", "prueba", "mercado"], within=pyo.NonNegativeReals)
d.obj = pyo.Objective(expr=100 * d.y["mecanizado"] + 80 * d.y["prueba"] + 40 * d.y["mercado"], sense=pyo.minimize)
d.valvula = pyo.Constraint(expr=2 * d.y["mecanizado"] + d.y["prueba"] + d.y["mercado"] >= 300)
d.conector = pyo.Constraint(expr=d.y["mecanizado"] + d.y["prueba"] >= 200)
pyo.SolverFactory("appsi_highs").solve(d)

print("Precio por unidad de recurso:", {k: pyo.value(d.y[k]) for k in d.y})
print("Pago total mínimo:", eur(pyo.value(d.obj)))

**Dos cosas que preguntan en entrevistas:**
1. Los precios del dual son **los mismos precios sombra** de la sección 2: 100, 100 y 0.
2. El pago mínimo (18.000 €) es **igual** a la ganancia máxima del taller. Esto se llama **dualidad fuerte** (*strong duality*).

Cualquier precio que cumpla las restricciones del dual da una cota: el taller nunca gana más que ese pago
(**dualidad débil**). Así sabe el solver que llegó al óptimo: la ganancia y la cota se tocan.
En la lección 3 verás la misma idea con el **gap** de los modelos enteros.

## 5. Costo reducido: lo que le falta a un producto para entrar
En el ejercicio 1 de la lección 1 agregaste la **brida** (1 h de mecanizado, 2 h de prueba, 250 €) y no entró al plan.
Calculaste a mano que "cuesta" 1×100 + 2×100 = 300 € de horas y solo deja 250 €.

Ese -50 € es el **costo reducido**. El solver lo da en `m.rc`.

**Analogía.** Un restaurante nuevo cerca de casa. Para que lo elijas en vez de tu favorito,
le falta 50 € de calidad. Si mejora menos que eso, sigues yendo al de siempre.

In [ ]:
g3 = {**GANANCIA, "brida": 250}
h3 = {**HORAS, ("brida", "mecanizado"): 1, ("brida", "prueba"): 2}
m3 = taller(ganancia=g3, horas=h3)

pd.DataFrame([{"producto": p, "unidades": pyo.value(m3.x[p]), "ganancia_unit": g3[p], "costo_reducido": m3.rc[m3.x[p]] + 0.0}
              for p in m3.P])

**Cómo se lee:**
- Válvula y conector se fabrican: su costo reducido es 0.
- Brida: -50 €. Para entrar al plan, su ganancia tiene que subir más de 50 € (a más de 300 €).

Es la misma holgura complementaria, del lado de los productos: si un producto se fabrica, su costo reducido es 0.

Comprobémoslo con la brida a 290 € y a 310 €.

In [ ]:
for precio in [290, 310]:
    mb = taller(ganancia={**GANANCIA, "brida": precio}, horas=h3)
    print(f"brida a {precio} € → plan {({p: round(pyo.value(mb.x[p]), 1) + 0.0 for p in mb.P})}, ganancia {eur(pyo.value(mb.obj))}")

## 6. El modelo crece: 30 productos × 8 máquinas
Con 2 productos lees todo a ojo. Con 30 productos y 8 máquinas ya no.
La estrategia nueva:
1. Los datos viven en **tablas** (como en SQL): una de productos, una de máquinas, una de horas.
2. El modelo se arma desde las tablas. El código es el mismo de la sección 2.
3. La solución vuelve a tablas y miramos el **top 5** de cada pieza.

Los datos son simulados con una semilla fija: siempre salen iguales.

In [ ]:
rng = np.random.default_rng(7)
maquinas = ["corte", "torno", "fresado", "soldadura", "tratamiento_termico", "pintura", "prueba_presion", "empaque"]
productos = [f"P{i:02d}" for i in range(1, 31)]

df_maq = pd.DataFrame({"maquina": maquinas, "capacidad_h": [400, 350, 300, 320, 250, 380, 280, 450]})

# Formato largo: una fila por (producto, máquina) que de verdad se usa. Cada producto pasa por la mitad de las máquinas.
df_horas = pd.DataFrame(
    [(p, k, round(rng.uniform(0.2, 3.0), 1)) for p in productos for k in maquinas if rng.random() < 0.5],
    columns=["producto", "maquina", "horas"],
)
df_prod = pd.DataFrame({"producto": productos,
                        "ganancia": rng.integers(80, 500, 30),
                        "demanda_max": rng.integers(20, 120, 30)})

print(df_prod.shape, df_maq.shape, df_horas.shape)
df_horas.head()

In [ ]:
df_prod.head()

In [ ]:
df_maq

### 6.1 De tablas a diccionarios
Pyomo lee diccionarios. `set_index(...)[col].to_dict()` convierte una tabla en `{clave: valor}`.

In [ ]:
ganancia = df_prod.set_index("producto")["ganancia"].to_dict()
demanda = df_prod.set_index("producto")["demanda_max"].to_dict()
capacidad = df_maq.set_index("maquina")["capacidad_h"].to_dict()
horas = df_horas.set_index(["producto", "maquina"])["horas"].to_dict()

print("ganancia['P01'] =", ganancia["P01"])
print("horas de P01:", {k: v for k, v in horas.items() if k[0] == "P01"})

### 6.2 El modelo
Mismas piezas que el taller. La restricción de mercado ahora es una por producto.

In [ ]:
def planta(ganancia, horas, capacidad, demanda):
    m = pyo.ConcreteModel()
    m.P = pyo.Set(initialize=list(ganancia))
    m.M = pyo.Set(initialize=list(capacidad))
    m.x = pyo.Var(m.P, within=pyo.NonNegativeReals)
    m.obj = pyo.Objective(expr=sum(ganancia[p] * m.x[p] for p in m.P), sense=pyo.maximize)
    m.cap = pyo.Constraint(m.M, rule=lambda m, k: sum(horas.get((p, k), 0) * m.x[p] for p in m.P) <= capacidad[k])
    m.dem = pyo.Constraint(m.P, rule=lambda m, p: m.x[p] <= demanda[p])
    m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    m.rc = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    resultado = pyo.SolverFactory("appsi_highs").solve(m)
    print("Estado:", resultado.solver.termination_condition, "| ganancia:", eur(pyo.value(m.obj)))
    return m


mp = planta(ganancia, horas, capacidad, demanda)

### 6.3 La solución, de vuelta a tablas
Dos tablas: una por máquina y una por producto. Cada una junta datos, solución y valores duales.

In [ ]:
res_maq = pd.DataFrame([{
    "maquina": k,
    "usado_h": pyo.value(mp.cap[k].body),
    "capacidad_h": capacidad[k],
    "holgura_h": capacidad[k] - pyo.value(mp.cap[k].body),
    "precio_sombra": mp.dual[mp.cap[k]],
} for k in mp.M]).round(1)

res_prod = pd.DataFrame([{
    "producto": p,
    "unidades": pyo.value(mp.x[p]),
    "demanda_max": demanda[p],
    "ganancia_unit": ganancia[p],
    "ganancia_total": ganancia[p] * pyo.value(mp.x[p]),
    "costo_reducido": mp.rc[mp.x[p]] + 0.0,  # + 0.0 quita el "-0.0"
    "valor_demanda": mp.dual[mp.dem[p]],  # precio sombra de la restricción de mercado
} for p in mp.P]).round(1)

print("Productos que se fabrican:", (res_prod["unidades"] > 0).sum(), "de", len(res_prod))
print("Productos al tope de demanda:", (res_prod["unidades"] == res_prod["demanda_max"]).sum())

### 6.4 Top 5 de cada pieza
**¿Dónde se gana la plata?** Los 5 productos que más aportan.

In [ ]:
res_prod.nlargest(5, "ganancia_total")[["producto", "unidades", "demanda_max", "ganancia_total"]]

**¿Qué máquina es el cuello de botella más caro?** Ordenadas por precio sombra.

In [ ]:
res_maq.sort_values("precio_sombra", ascending=False)

**¿Qué producto está más cerca de entrar al plan?** Los no fabricados con el costo reducido más cercano a 0.

In [ ]:
fuera = res_prod[res_prod["unidades"] == 0]
fuera.nlargest(5, "costo_reducido")[["producto", "ganancia_unit", "costo_reducido"]]

**¿En qué producto vale más vender una unidad extra?** El precio sombra de su límite de demanda.

In [ ]:
res_prod.nlargest(5, "valor_demanda")[["producto", "unidades", "demanda_max", "valor_demanda"]]

**Lectura para el jefe de planta:**
- 5 máquinas están al 100 %. La más cara es **prueba de presión**: cada hora extra vale unos 171 €.
- Corte, torno y empaque tienen holgura: una hora más vale 0. No hay que invertir ahí.
- **P04** está a 8 € por unidad de entrar al plan. Una pequeña subida de precio lo mete.
- Una unidad más de demanda de **P23** vale 255 €. Si marketing la consigue por menos, conviene.

## 7. Una decisión real, comprobada
Un proveedor ofrece horas de prueba de presión a 120 €/h. El precio sombra es 171,5 €/h: parece buen negocio.
Pero el precio sombra vale **dentro de un rango**. Comprobemos con 20 h y con 100 h extra.

In [ ]:
base = pyo.value(mp.obj)
precio_sombra = mp.dual[mp.cap["prueba_presion"]]
for extra in [20, 100]:
    mx = planta(ganancia, horas, {**capacidad, "prueba_presion": capacidad["prueba_presion"] + extra}, demanda)
    real = pyo.value(mx.obj) - base
    print(f"  +{extra} h: predicho {eur(extra * precio_sombra)}, real {eur(real)}, costo {eur(extra * 120)}")

**Lectura:**
- Con 20 h, lo predicho y lo real coinciden: estamos dentro del rango.
- Con 100 h, lo real es menor que lo predicho: nos salimos del rango y otra máquina empezó a mandar.
- En los dos casos la compra conviene (gana más de lo que cuesta), pero el número que le das al jefe para 100 h es el real, no el predicho.

**Regla de trabajo:** el precio sombra te dice **dónde mirar**. Un re-solve te dice **cuánto**.

## 8. Una advertencia: esto es para LP
El plan tiene unidades con decimales (43,3 de P01). En un programa lineal está permitido.
Si exiges unidades enteras, el modelo pasa a ser un **MILP** y los precios sombra dejan de estar bien definidos:
la ganancia ya no cambia de forma suave, sino a saltos.

En la práctica: para sensibilidad en un MILP se fijan las variables enteras y se resuelve el LP que queda,
o se hacen re-solves con escenarios. Eso viene en la lección 3.

## 9. Resumen: conceptos de hoy
| Concepto | Qué es | En el taller | En inglés |
|---|---|---|---|
| Precio sombra | Cuánto sube el objetivo si el lado derecho de una restricción sube 1 | mecanizado: 100 €/h | shadow price, dual value |
| Holgura complementaria | Con holgura, precio sombra 0. Si se fabrica, costo reducido 0 | mercado: holgura 20, precio 0 | complementary slackness |
| Rango | Intervalo donde el precio sombra no cambia | mecanizado: de 80 a 120 h | RHS ranging, allowable increase/decrease |
| Costo reducido | Cuánto debe mejorar un producto para entrar al plan | brida: -50 € | reduced cost |
| Problema dual | El mismo modelo visto como precios de los recursos | el vecino que compra tus horas | dual problem |
| Dualidad fuerte | En el óptimo, primal y dual valen lo mismo | 18.000 € = 18.000 € | strong duality |
| Dualidad débil | Cualquier solución dual es una cota del primal | — | weak duality |

## 10. Ejercicios
1. **El taller con más mecanizado.** Usa `taller(capacidad={"mecanizado": 130, "prueba": 80})` y mira `tabla_restricciones`.
   ¿Qué precio sombra tiene cada restricción? ¿Qué te dice el precio del mercado?
2. **Que entre P04.** Según su costo reducido, ¿cuánto debe subir la ganancia de P04 para que se fabrique? Comprueba con `planta`.
3. **Campaña de marketing.** Cuesta 2.000 € y sube 10 unidades la demanda de P23. ¿Vale la pena? Responde primero con el precio sombra y después comprueba.

In [ ]:
# Escribe aquí tus intentos

## En la entrevista
Pregunta típica: *"What is a shadow price and how would you use it?"*
> *"It is the change in the optimal objective per unit increase in a constraint's right-hand side, valid only within its range.
> If a constraint has slack, its shadow price is zero: complementary slackness. The reduced cost tells me how much a product's
> margin must improve before it enters the plan. I use them to find where to look, and I confirm the number with a re-solve
> before I give it to the business. For a MILP I fix the integer variables and read the duals of the remaining LP."*

## Soluciones (no mires antes de intentar)

In [ ]:
# Ejercicio 1
tabla_restricciones(taller(capacidad={"mecanizado": 130, "prueba": 80}))

Con 130 h el mecanizado tiene holgura (usa 120 h): su precio sombra es 0. El plan es 40 válvulas y 40 conectores.
Prueba vale 200 €/h: una hora más es un conector más.
El mercado vale 100 €: una válvula más de demanda te deja cambiar un conector por una válvula (300 - 200 = 100 €).
Ahora el cuello de botella es el cliente, no la máquina.

In [ ]:
# Ejercicio 2
rc_p04 = res_prod.set_index("producto").loc["P04", "costo_reducido"]
print("Costo reducido de P04:", rc_p04)
for subida in [5, 10]:
    mx = planta({**ganancia, "P04": ganancia["P04"] + subida}, horas, capacidad, demanda)
    print(f"  P04 +{subida} € → {pyo.value(mx.x['P04']):.1f} unidades")

Le faltan 8,2 €. Con +5 € no entra; con +10 € entra con unas 20 unidades.

In [ ]:
# Ejercicio 3
valor = res_prod.set_index("producto").loc["P23", "valor_demanda"]
print("Predicho:", eur(10 * valor))
mx = planta(ganancia, horas, capacidad, {**demanda, "P23": demanda["P23"] + 10})
print("Real:", eur(pyo.value(mx.obj) - pyo.value(mp.obj)))

Predicho: 10 × 255,4 ≈ 2.554 €. Real: lo mismo, porque 10 unidades están dentro del rango.
Gana 2.554 € y cuesta 2.000 €: vale la pena, por unos 554 €.